# Grounded RAG Chatbot (Complex)

This notebook implements the same pipeline as `complex_app.py`:

1. Load FAQ documents with metadata (Category, Language, Version)
2. Q&A-aware chunking (with fixed-size fallback)
3. Embeddings via OpenRouter
4. Store in ChromaDB with metadata
5. Retrieve with top-k, similarity threshold, and category filters
6. Compress context and generate **grounded** answers (Answer / Evidence / Confidence)
7. Detect RAG failure modes, log queries, and run basic evaluation signals

**Setup:** Create a `.env` file with `OPENROUTER_API_KEY=your-key` ([OpenRouter](https://openrouter.ai/)) and install dependencies from `requirements.txt`.

Run cells in order. For the interactive demo, change `question` in section 15 and re-run.

## 1. Environment and imports

In [2]:
import os
import re
import csv
import uuid
import time
from datetime import datetime
from typing import List, Dict, Any

import pandas as pd
import chromadb
from dotenv import load_dotenv
from openai import OpenAI

In [3]:
load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if not OPENROUTER_API_KEY:
    raise ValueError(
        "OPENROUTER_API_KEY is missing. Get a key at https://openrouter.ai/ "
        "and add it to your .env file."
    )

client = OpenAI(
    base_url=OPENROUTER_BASE_URL,
    api_key=OPENROUTER_API_KEY,
)

## 2. Configuration

In [10]:
EMBEDDING_MODEL = "text-embedding-3-small"
GENERATION_MODEL = "gpt-4o-mini"


def openrouter_model(model: str) -> str:
    """Map short model names to OpenRouter provider/model slugs."""
    if "/" in model:
        return model
    return f"openai/{model}"


FAQ_FILE_PATH = "data/full_detailed_data.txt"
CHROMA_DB_PATH = "chroma_db"
COLLECTION_NAME = "grounded_faq_rag"

LOG_DIR = "logs"
QUERY_LOG_PATH = os.path.join(LOG_DIR, "query_log.csv")
FEEDBACK_LOG_PATH = os.path.join(LOG_DIR, "feedback_log.csv")

os.makedirs(LOG_DIR, exist_ok=True)

## 3. Initialize ChromaDB

In [11]:
chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)
print(f"Collection '{COLLECTION_NAME}' ready. Current chunk count: {collection.count()}")

Collection 'grounded_faq_rag' ready. Current chunk count: 0


## 4. Load documents

In [12]:
def load_text_file(file_path: str) -> str:
    """Loads a text document from disk."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    with open(file_path, "r", encoding="utf-8") as file:
        return file.read()


faq_text = load_text_file(FAQ_FILE_PATH)
print(f"Loaded {len(faq_text)} characters from {FAQ_FILE_PATH}")
print("\n--- Preview (first 600 chars) ---\n")
print(faq_text[:600])

Loaded 9190 characters from data/full_detailed_data.txt

--- Preview (first 600 chars) ---

Category: rag_basics
Language: en
Version: 2026-05

Q: What is a RAG chatbot?
A: A RAG chatbot retrieves relevant information from documents before generating an answer.

Q: What does RAG stand for?
A: RAG stands for Retrieval-Augmented Generation.

Q: What problem does RAG solve?
A: RAG helps the chatbot answer using relevant information instead of relying only on the model’s memory.

Q: What is the difference between RAG and a normal chatbot?
A: A normal chatbot answers from model knowledge, while a RAG chatbot first retrieves information from documents.

Q: What is the main goal of a basic 


## 5. Metadata extraction and Q&A chunking

In [13]:
def extract_metadata_from_block(block: str) -> Dict[str, Any]:
    metadata = {
        "source": FAQ_FILE_PATH,
        "category": "general",
        "language": "en",
        "version": "unknown",
    }

    category_match = re.search(r"Category:\s*(.+)", block, re.IGNORECASE)
    language_match = re.search(r"Language:\s*(.+)", block, re.IGNORECASE)
    version_match = re.search(r"Version:\s*(.+)", block, re.IGNORECASE)

    if category_match:
        metadata["category"] = category_match.group(1).strip()
    if language_match:
        metadata["language"] = language_match.group(1).strip()
    if version_match:
        metadata["version"] = version_match.group(1).strip()

    return metadata


def split_into_qa_chunks(text: str) -> List[Dict[str, Any]]:
    chunks = []
    sections = re.split(r"\n(?=Category:)", text)

    for section in sections:
        section = section.strip()
        if not section:
            continue

        metadata = extract_metadata_from_block(section)
        qa_pairs = re.findall(
            r"(Q:\s*.*?)(?=\nQ:|\Z)",
            section,
            flags=re.DOTALL | re.IGNORECASE,
        )

        for qa in qa_pairs:
            clean_chunk = qa.strip()
            if clean_chunk:
                chunks.append({"text": clean_chunk, "metadata": metadata})

    return chunks


def split_into_fallback_chunks(
    text: str,
    chunk_size: int = 600,
    overlap: int = 100,
) -> List[Dict[str, Any]]:
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk_text = text[start:end].strip()

        if chunk_text:
            chunks.append({
                "text": chunk_text,
                "metadata": {
                    "source": FAQ_FILE_PATH,
                    "category": "general",
                    "language": "en",
                    "version": "unknown",
                },
            })

        start += chunk_size - overlap

    return chunks


def create_chunks(text: str) -> List[Dict[str, Any]]:
    qa_chunks = split_into_qa_chunks(text)
    if qa_chunks:
        return qa_chunks
    return split_into_fallback_chunks(text)


chunks = create_chunks(faq_text)
print(f"Created {len(chunks)} chunks")
print("\n--- First chunk ---\n")
print(chunks[0]["text"][:400])
print("\n--- Metadata ---\n")
print(chunks[0]["metadata"])

Created 70 chunks

--- First chunk ---

Q: What is a RAG chatbot?
A: A RAG chatbot retrieves relevant information from documents before generating an answer.

--- Metadata ---

{'source': 'data/full_detailed_data.txt', 'category': 'rag_basics', 'language': 'en', 'version': '2026-05'}


## 6. Create embeddings

In [14]:
def get_embedding(text: str) -> List[float]:
    """Creates an embedding vector via OpenRouter."""
    response = client.embeddings.create(
        model=openrouter_model(EMBEDDING_MODEL),
        input=text,
    )
    return response.data[0].embedding


sample_embedding = get_embedding(chunks[0]["text"][:200])
print(f"Embedding dimension: {len(sample_embedding)}")

Embedding dimension: 1536


## 7. Ingest chunks into the vector database

Run once. Skips if documents are already ingested. Set `FORCE_REINGEST = True` to rebuild.

In [15]:
def clear_collection() -> None:
    global collection
    try:
        chroma_client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass
    collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)


def ingest_documents(force_reingest: bool = False) -> str:
    if force_reingest:
        clear_collection()

    if collection.count() > 0:
        return f"Documents already ingested. Current chunks: {collection.count()}"

    text = load_text_file(FAQ_FILE_PATH)
    doc_chunks = create_chunks(text)

    ids = []
    documents = []
    embeddings = []
    metadatas = []

    for index, chunk in enumerate(doc_chunks):
        metadata = chunk["metadata"].copy()
        metadata["chunk_index"] = index
        metadata["ingested_at"] = datetime.now().isoformat()

        ids.append(str(uuid.uuid4()))
        documents.append(chunk["text"])
        embeddings.append(get_embedding(chunk["text"]))
        metadatas.append(metadata)

    collection.add(
        ids=ids,
        documents=documents,
        embeddings=embeddings,
        metadatas=metadatas,
    )

    return f"Ingestion completed. Stored {len(doc_chunks)} chunks."


FORCE_REINGEST = True
print(ingest_documents(force_reingest=FORCE_REINGEST))

Ingestion completed. Stored 70 chunks.


## 8. Context optimization

In [16]:
def deduplicate_chunks(chunks_list: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = set()
    unique_chunks = []

    for chunk in chunks_list:
        text = chunk["text"].strip()
        if text not in seen:
            seen.add(text)
            unique_chunks.append(chunk)

    return unique_chunks


def compress_context(chunks_list: List[Dict[str, Any]], max_chars: int = 3000) -> str:
    context_parts = []
    current_length = 0

    for i, chunk in enumerate(chunks_list, start=1):
        source = chunk["metadata"].get("source", "unknown")
        category = chunk["metadata"].get("category", "general")
        version = chunk["metadata"].get("version", "unknown")
        chunk_text = chunk["text"].strip()

        formatted_chunk = f"""
[Context {i}]
Source: {source}
Category: {category}
Version: {version}
Content:
{chunk_text}
""".strip()

        if current_length + len(formatted_chunk) > max_chars:
            break

        context_parts.append(formatted_chunk)
        current_length += len(formatted_chunk)

    return "\n\n".join(context_parts)

## 9. Retrieval tuning

In [17]:
def retrieve_relevant_chunks(
    question: str,
    top_k: int = 3,
    similarity_threshold: float = 0.35,
    category_filter: str = "All",
) -> List[Dict[str, Any]]:
    question_embedding = get_embedding(question)

    where_filter = None
    if category_filter and category_filter != "All":
        where_filter = {"category": category_filter}

    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=top_k,
        where=where_filter,
        include=["documents", "metadatas", "distances"],
    )

    retrieved = []
    documents = results.get("documents", [[]])[0]
    metadatas = results.get("metadatas", [[]])[0]
    distances = results.get("distances", [[]])[0]

    for doc, metadata, distance in zip(documents, metadatas, distances):
        similarity_score = 1 - distance
        if similarity_score >= similarity_threshold:
            retrieved.append({
                "text": doc,
                "metadata": metadata,
                "distance": distance,
                "similarity_score": similarity_score,
            })

    return deduplicate_chunks(retrieved)

## 10. Grounded prompt and answer generation

In [18]:
def build_grounded_prompt(question: str, context: str) -> str:
    return f"""
You are a grounded FAQ assistant.

Your role:
Answer user questions using only the retrieved context.

Source rule:
Use only the context provided below.
Do not use outside knowledge.
Do not guess.
Do not invent facts.
Do not answer beyond the provided context.

Missing context rule:
If the answer is not clearly supported by the context, say:
"I don't know based on the provided documents."

Partial support rule:
If the context only partially answers the question:
- Answer only the supported part.
- Clearly say what information is missing.

Consistency rule:
Always use this answer format:

Answer:
<short answer>

Evidence:
- Source: <source>
- Relevant text: <short evidence text>

Confidence:
High / Medium / Low

Retrieved context:
{context}

User question:
{question}
""".strip()


def generate_grounded_answer(
    question: str,
    retrieved_chunks: List[Dict[str, Any]],
    temperature: float = 0.0,
) -> str:
    if not retrieved_chunks:
        return """
Answer:
I don't know based on the provided documents.

Evidence:
- Source: No relevant source found.
- Relevant text: No relevant context was retrieved.

Confidence:
Low
""".strip()

    context = compress_context(retrieved_chunks)
    prompt = build_grounded_prompt(question=question, context=context)

    response = client.chat.completions.create(
        model=openrouter_model(GENERATION_MODEL),
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
    )

    return response.choices[0].message.content

## 11. RAG failure detection, logging, and evaluation

In [19]:
def detect_rag_failure(answer: str, retrieved_chunks: List[Dict[str, Any]]) -> List[str]:
    failures = []

    if not retrieved_chunks:
        failures.append("Missing answer: no relevant chunks retrieved.")
    if "I don't know based on the provided documents" in answer:
        failures.append("Refusal triggered: answer not found in retrieved context.")
    if len(answer) > 2000:
        failures.append("Long answer: possible too much context or weak prompt control.")
    if "Evidence:" not in answer:
        failures.append("Inconsistent format: answer does not include evidence section.")
    if "Confidence:" not in answer:
        failures.append("Inconsistent format: answer does not include confidence section.")

    return failures


def append_csv_row(file_path: str, row: Dict[str, Any]) -> None:
    file_exists = os.path.exists(file_path)
    with open(file_path, "a", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=row.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


def log_query(
    question: str,
    answer: str,
    retrieved_chunks: List[Dict[str, Any]],
    latency: float,
    failures: List[str],
) -> None:
    sources = [c["metadata"].get("source", "unknown") for c in retrieved_chunks]
    categories = [c["metadata"].get("category", "general") for c in retrieved_chunks]
    similarities = [round(c["similarity_score"], 4) for c in retrieved_chunks]

    row = {
        "timestamp": datetime.now().isoformat(),
        "question": question,
        "answer": answer,
        "num_retrieved_chunks": len(retrieved_chunks),
        "sources": " | ".join(sources),
        "categories": " | ".join(categories),
        "similarity_scores": " | ".join(map(str, similarities)),
        "latency_seconds": round(latency, 3),
        "failures": " | ".join(failures),
    }
    append_csv_row(QUERY_LOG_PATH, row)


def evaluate_single_response(
    question: str,
    answer: str,
    retrieved_chunks: List[Dict[str, Any]],
) -> Dict[str, Any]:
    answer_text = answer.lower()
    has_context = len(retrieved_chunks) > 0
    has_evidence = "evidence:" in answer_text
    has_confidence = "confidence:" in answer_text
    refused = "i don't know based on the provided documents" in answer_text

    if has_context and not refused:
        groundedness = "Needs human review"
    elif refused:
        groundedness = "Safe refusal"
    else:
        groundedness = "No context retrieved"

    return {
        "question": question,
        "has_context": has_context,
        "has_evidence_section": has_evidence,
        "has_confidence_section": has_confidence,
        "refused_when_missing": refused,
        "groundedness_signal": groundedness,
    }

## 12. Full RAG pipeline (ask a question)

Adjust `question`, `top_k`, `similarity_threshold`, `category_filter`, and `temperature`, then re-run.

In [26]:
question = "How does RAG chatbot work, and where does it gets its info?"

top_k = 7
similarity_threshold = 0.35
category_filter = "All"  # e.g. "rag_basics", "embeddings", or "All"
temperature = 1

if collection.count() == 0:
    print("Run the ingestion cell first (section 7).")
else:
    start_time = time.time()

    retrieved_chunks = retrieve_relevant_chunks(
        question=question,
        top_k=top_k,
        similarity_threshold=similarity_threshold,
        category_filter=category_filter,
    )

    answer = generate_grounded_answer(
        question=question,
        retrieved_chunks=retrieved_chunks,
        temperature=temperature,
    )

    latency = time.time() - start_time
    failures = detect_rag_failure(answer, retrieved_chunks)
    log_query(question, answer, retrieved_chunks, latency, failures)
    evaluation = evaluate_single_response(question, answer, retrieved_chunks)

    print("Question:", question)
    print(f"Latency: {latency:.2f}s")
    print("\n--- Answer ---\n")
    print(answer)

    if failures:
        print("\n--- RAG failure signals ---")
        for f in failures:
            print("-", f)

    print("\n--- Evaluation ---")
    print(evaluation)

    print("\n--- Retrieved context ---")
    for i, chunk in enumerate(retrieved_chunks, start=1):
        print(f"\nChunk {i} (similarity={chunk['similarity_score']:.4f})")
        print(chunk["text"][:500])
        print("Metadata:", chunk["metadata"])

Question: How does RAG chatbot work, and where does it gets its info?
Latency: 4.39s

--- Answer ---

Answer:
A RAG chatbot works by embedding the user's question, retrieving relevant chunks from documents, and then using those chunks to generate an answer.

Evidence:
- Source: data/full_detailed_data.txt
- Relevant text: "The system embeds the question, retrieves relevant chunks, and asks the model to answer using those chunks."

Confidence:
High

--- Evaluation ---
{'question': 'How does RAG chatbot work, and where does it gets its info?', 'has_context': True, 'has_evidence_section': True, 'has_confidence_section': True, 'refused_when_missing': False, 'groundedness_signal': 'Needs human review'}

--- Retrieved context ---

Chunk 1 (similarity=0.5896)
Q: What is a RAG chatbot?
A: A RAG chatbot retrieves relevant information from documents before generating an answer.
Metadata: {'language': 'en', 'chunk_index': 0, 'source': 'data/full_detailed_data.txt', 'version': '2026-05', 'category

In [ ]:
question = "What is a RAG chatbot?"

top_k = 3
similarity_threshold = 0.35
category_filter = "All"  # e.g. "rag_basics", "embeddings", or "All"
temperature = 0.0

if collection.count() == 0:
    print("Run the ingestion cell first (section 7).")
else:
    start_time = time.time()

    retrieved_chunks = retrieve_relevant_chunks(
        question=question,
        top_k=top_k,
        similarity_threshold=similarity_threshold,
        category_filter=category_filter,
    )

    answer = generate_grounded_answer(
        question=question,
        retrieved_chunks=retrieved_chunks,
        temperature=temperature,
    )

    latency = time.time() - start_time
    failures = detect_rag_failure(answer, retrieved_chunks)
    log_query(question, answer, retrieved_chunks, latency, failures)
    evaluation = evaluate_single_response(question, answer, retrieved_chunks)

    print("Question:", question)
    print(f"Latency: {latency:.2f}s")
    print("\n--- Answer ---\n")
    print(answer)

    if failures:
        print("\n--- RAG failure signals ---")
        for f in failures:
            print("-", f)

    print("\n--- Evaluation ---")
    print(evaluation)

    print("\n--- Retrieved context ---")
    for i, chunk in enumerate(retrieved_chunks, start=1):
        print(f"\nChunk {i} (similarity={chunk['similarity_score']:.4f})")
        print(chunk["text"][:500])
        print("Metadata:", chunk["metadata"])

## 13. View query logs

In [27]:
if os.path.exists(QUERY_LOG_PATH):
    pd.read_csv(QUERY_LOG_PATH)
else:
    print("No query logs yet. Run section 12 first.")

### Optional: reset the vector database

Uncomment and run if you need to re-ingest (e.g. after changing the FAQ file).

In [ ]:
# clear_collection()
# print(ingest_documents(force_reingest=True))

### OpenRouter API test

Run after sections 1–2 to verify your `OPENROUTER_API_KEY`.

In [ ]:
try:
    chat_response = client.chat.completions.create(
        model=openrouter_model(GENERATION_MODEL),
        messages=[{"role": "user", "content": "Say: OpenRouter API key is working."}],
        max_tokens=30,
    )
    print("Chat test passed:", chat_response.choices[0].message.content)

    embedding_response = client.embeddings.create(
        model=openrouter_model(EMBEDDING_MODEL),
        input="This is a test sentence for embeddings.",
    )
    vector = embedding_response.data[0].embedding
    print("Embedding test passed. Vector length:", len(vector))

except Exception as e:
    print("OpenRouter API test failed.")
    print("Error:", e)